In [13]:
import numpy as np
import pandas as pd
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from matplotlib import pyplot
import imblearn
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn import datasets
from sklearn.preprocessing import LabelBinarizer

In [14]:
df = pd.read_excel("/content/perfect_final_data_for_multilevel_analysis (1).xlsx")
df

,V001,V002,V012,V013,V024,V025,V106,V113,V130,V157,...,Number_of_children,Living_children,Age_of_husband,profession_of_husband,Birth_Interval,Drinking_water,Wealth_Status,Religion_status,BMI,Respondent_Age_first_birth
0,469,57,48,7,6,2,0,21,1,0,...,2,2,2,1,4,2,0,1,4,1
1,438,26,32,4,6,1,2,21,1,0,...,1,1,1,2,1,2,2,1,4,0
2,296,96,39,5,4,1,2,21,1,0,...,2,2,2,4,1,2,2,1,4,1
3,438,4,48,7,6,1,0,21,1,0,...,2,2,2,2,1,2,2,1,4,0
4,548,158,32,4,7,2,3,12,1,0,...,1,1,1,1,2,1,0,1,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8497,417,132,41,6,5,2,1,21,1,0,...,2,1,2,1,3,2,0,1,1,0
8498,190,11,43,6,3,1,0,11,1,0,...,1,1,1,3,3,1,1,1,1,1
8499,123,123,40,6,2,2,0,21,1,0,...,1,1,2,2,1,2,0,1,1,1
8500,581,64,30,4,7,2,1,21,2,0,...,2,2,1,2,2,2,0,0,1,1


In [15]:
df.drop(columns=['V001', 'V002', 'V012', 'V113', 'V130', 'V157', 'V158',
                 'V159', 'V190', 'V201', 'V212', 'V218', 'V221', 'V364', 'V445','V447', 'V501',
                 'V502', 'V701', 'V705', 'V730'], inplace=True)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8502 entries, 0 to 8501
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   V013                        8502 non-null   int64  
 1   V024                        8502 non-null   int64  
 2   V025                        8502 non-null   int64  
 3   V106                        8502 non-null   int64  
 4   V404                        8502 non-null   int64  
 5   V714                        8502 non-null   int64  
 6   BMI_fixed                   8502 non-null   float64
 7   Media_acccess               8502 non-null   int64  
 8   husband_education           8502 non-null   int64  
 9   contraceptive_use           8502 non-null   int64  
 10  Number_of_children          8502 non-null   int64  
 11  Living_children             8502 non-null   int64  
 12  Age_of_husband              8502 non-null   int64  
 13  profession_of_husband       8502 

In [17]:
df.drop(columns=['BMI_fixed'], inplace=True)

In [18]:
#Log Scaling (FT3)
from sklearn.preprocessing import FunctionTransformer
transformer=FunctionTransformer(np.log1p)
t1=transformer.transform(df)
print(t1)

          V013      V024      V025      V106      V404      V714  \
0     2.079442  1.945910  1.098612  0.000000  0.000000  0.000000   
1     1.609438  1.945910  0.693147  1.098612  0.000000  0.693147   
2     1.791759  1.609438  0.693147  1.098612  0.000000  0.000000   
3     2.079442  1.945910  0.693147  0.000000  0.000000  0.000000   
4     1.609438  2.079442  1.098612  1.386294  0.000000  0.000000   
...        ...       ...       ...       ...       ...       ...   
8497  1.945910  1.791759  1.098612  0.693147  0.000000  0.000000   
8498  1.945910  1.386294  0.693147  0.000000  0.000000  0.000000   
8499  1.945910  1.098612  1.098612  0.000000  0.000000  0.000000   
8500  1.609438  2.079442  1.098612  0.693147  0.693147  0.000000   
8501  1.791759  1.098612  0.693147  0.000000  0.000000  0.000000   

      Media_acccess  husband_education  contraceptive_use  Number_of_children  \
0          0.000000           0.000000           0.000000            1.098612   
1          0.693147  

In [19]:

y = t1.BMI  # use bmi1 as the target variable
x = t1.drop('BMI', axis=1)  # use the other columns as features

In [20]:
x  # display the feature columns

,V013,V024,V025,V106,V404,V714,Media_acccess,husband_education,contraceptive_use,Number_of_children,Living_children,Age_of_husband,profession_of_husband,Birth_Interval,Drinking_water,Wealth_Status,Religion_status,Respondent_Age_first_birth
0,2.079442,1.945910,1.098612,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.098612,1.098612,1.098612,0.693147,1.609438,1.098612,0.000000,0.693147,0.693147
1,1.609438,1.945910,0.693147,1.098612,0.000000,0.693147,0.693147,1.098612,0.000000,0.693147,0.693147,0.693147,1.098612,0.693147,1.098612,1.098612,0.693147,0.000000
2,1.791759,1.609438,0.693147,1.098612,0.000000,0.000000,0.693147,1.386294,0.693147,1.098612,1.098612,1.098612,1.609438,0.693147,1.098612,1.098612,0.693147,0.693147
3,2.079442,1.945910,0.693147,0.000000,0.000000,0.000000,0.693147,1.098612,0.000000,1.098612,1.098612,1.098612,1.098612,0.693147,1.098612,1.098612,0.693147,0.000000
4,1.609438,2.079442,1.098612,1.386294,0.000000,0.000000,0.000000,0.693147,0.000000,0.693147,0.693147,0.693147,0.693147,1.098612,0.693147,0.000000,0.693147,0.693147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8497,1.945910,1.791759,1.098612,0.693147,0.000000,0.000000,0.000000,0.000000,0.693147,1.098612,0.693147,1.098612,0.693147,1.386294,1.098612,0.000000,0.693147,0.000000
8498,1.945910,1.386294,0.693147,0.000000,0.000000,0.000000,0.693147,0.000000,0.000000,0.693147,0.693147,0.693147,1.386294,1.386294,0.693147,0.693147,0.693147,0.693147
8499,1.945910,1.098612,1.098612,0.000000,0.000000,0.000000,0.000000,0.000000,0.693147,0.693147,0.693147,1.098612,1.098612,0.693147,1.098612,0.000000,0.693147,0.693147
8500,1.609438,2.079442,1.098612,0.693147,0.693147,0.000000,0.693147,0.693147,0.693147,1.098612,1.098612,0.693147,1.098612,1.098612,1.098612,0.000000,0.000000,0.693147


In [21]:
y = LabelEncoder().fit_transform(y)  # change target labels into numbers
oversample = SMOTE()  # create SMOTE to balance classes
x, y = oversample.fit_resample(x, y)  # make class counts more balanced
counter = Counter(y)  # count samples in each class
for k, v in counter.items():  # loop through each class count
  per = v / len(y) * 100  # calculate class percentage
  print('class=%d, count=%d, percentage=%.3f%%' % (k, v, per))  # print class distribution

class=3, count=4469, percentage=25.000%
class=2, count=4469, percentage=25.000%
class=1, count=4469, percentage=25.000%
class=0, count=4469, percentage=25.000%


In [22]:
from sklearn.model_selection import train_test_split  # import the train-test split tool
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=101)  # split data into training and testing sets
x_train.head()  # preview the training features
x_train.shape  # check the training data shape

(12513, 18)

In [23]:
x_test.head()  # preview the testing features
x_test.shape  # check the testing data shape

(5363, 18)

In [24]:
# logistic regression
from sklearn.linear_model import LogisticRegression  # import logistic regression
Lr = LogisticRegression()  # create the model
Lr.fit(x_train, y_train)  # train the model on the training data

# Generate predictions for both test and training data
predictions = Lr.predict(x_test)  # predict labels for the test set
train_predictions = Lr.predict(x_train)  # predict labels for the training set

from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools

# Train performance
print("=== Training Performance ===")  # print training results heading
print(classification_report(y_train, train_predictions, digits=4))  # show training metrics
print(confusion_matrix(y_train, train_predictions))  # show training confusion matrix
# test performance
print("=== Test Performance ===")  # print test results heading
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = Lr.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities for ROC work

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # measure agreement between true and predicted labels

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(Lr, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Training Performance ===
              precision    recall  f1-score   support

           0     0.4682    0.6032    0.5272      3120
           1     0.3237    0.2168    0.2597      3113
           2     0.3387    0.2179    0.2652      3112
           3     0.4573    0.6360    0.5321      3168

    accuracy                         0.4196     12513
   macro avg     0.3970    0.4185    0.3960     12513
weighted avg     0.3973    0.4196    0.3967     12513

[[1882  563  304  371]
 [1173  675  527  738]
 [ 622  530  678 1282]
 [ 343  317  493 2015]]
=== Test Performance ===
              precision    recall  f1-score   support

           0     0.4463    0.5849    0.5063      1349
           1     0.2978    0.2021    0.2408      1356
           2     0.3257    0.2100    0.2554      1357
           3     0.4372    0.6049    0.5076      1301

    accuracy                         0.3981      5363
   macro avg     0.3768    0.4005    0.3775      5363
weighted avg     0.3760    0.3981    0

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Cross Validation Scores are [0.41051454 0.41219239 0.40604027 0.41498881 0.40324385 0.42114094
 0.39563514 0.40682708 0.38332401 0.42697258 0.43120805 0.41275168
 0.42002237 0.38814318 0.41722595 0.39597315 0.41186346 0.38108562
 0.40235031 0.41466144 0.4189038  0.41387025 0.40212528 0.41778523
 0.3942953  0.39765101 0.41298265 0.40794628 0.41130386 0.41130386]
Average Cross Validation score :0.4081444119369028


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier  # import decision tree model
dtree = DecisionTreeClassifier()  # create the model
dtree.fit(x_train, y_train)  # train the model
predictions = dtree.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = dtree.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(dtree, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.6834    0.6976    0.6904      1349
           1     0.5408    0.5133    0.5267      1356
           2     0.4644    0.4562    0.4602      1357
           3     0.6933    0.7279    0.7102      1301

    accuracy                         0.5972      5363
   macro avg     0.5954    0.5987    0.5969      5363
weighted avg     0.5943    0.5972    0.5956      5363

[[941 150 175  83]
 [169 696 363 128]
 [182 348 619 208]
 [ 85  93 176 947]]
Cross Validation Scores are [0.59731544 0.598434   0.62751678 0.61241611 0.60794183 0.61297539
 0.60716284 0.60324566 0.61667599 0.59373251 0.61744966 0.60346756
 0.5917226  0.62751678 0.60234899 0.60961969 0.60716284 0.60548405
 0.62618914 0.58533856 0.60234899 0.60850112 0.60178971 0.60794183
 0.60794183 0.62024609 0.6144376  0.60604365 0.61163962 0.60604365]
Average Cross Validation score :0.6080216844915659


In [26]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier  # import random forest model
rfc = RandomForestClassifier(n_estimators=100)  # create the model with 100 trees
rfc.fit(x_train, y_train)  # train the model
predictions = rfc.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = rfc.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(rfc, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.8473    0.8347    0.8409      1349
           1     0.5925    0.7301    0.6541      1356
           2     0.6499    0.5144    0.5742      1357
           3     0.8565    0.8486    0.8525      1301

    accuracy                         0.7306      5363
   macro avg     0.7365    0.7319    0.7304      5363
weighted avg     0.7351    0.7306    0.7290      5363

[[1126  151   35   37]
 [  60  990  271   35]
 [ 104  442  698  113]
 [  39   88   70 1104]]
Cross Validation Scores are [0.74888143 0.76789709 0.7639821  0.75223714 0.75223714 0.75
 0.75209849 0.74762171 0.7313934  0.75825406 0.7533557  0.7533557
 0.74105145 0.75559284 0.75894855 0.75391499 0.73922776 0.74482373
 0.74258534 0.74090655 0.74272931 0.75503356 0.74608501 0.77628635
 0.74328859 0.74272931 0.7308338  0.74818131 0.76161164 0.7504197 ]
Average Cross Validation score :0.7501854578201087


In [27]:
# K- nearest neighbor
from sklearn.neighbors import KNeighborsClassifier  # import KNN model
KN = KNeighborsClassifier()  # create the model
KN.fit(x_train, y_train)  # train the model
predictions = KN.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = KN.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(KN, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.6505    0.9548    0.7738      1349
           1     0.5652    0.2684    0.3640      1356
           2     0.6400    0.4886    0.5541      1357
           3     0.7275    0.9523    0.8249      1301

    accuracy                         0.6627      5363
   macro avg     0.6458    0.6660    0.6292      5363
weighted avg     0.6450    0.6627    0.6270      5363

[[1288   25   23   13]
 [ 428  364  333  231]
 [ 238  236  663  220]
 [  26   19   17 1239]]
Cross Validation Scores are [0.70973154 0.70917226 0.70134228 0.6901566  0.70749441 0.71029083
 0.70285395 0.71516508 0.71796307 0.70285395 0.69854586 0.71420582
 0.70581655 0.71085011 0.71085011 0.71420582 0.69893677 0.7112479
 0.70509233 0.70229435 0.70525727 0.70973154 0.69519016 0.70022371
 0.70022371 0.71644295 0.70173475 0.70061556 0.70117515 0.68830442]
Average Cross Validation score :0.7052656270930119


In [28]:
# GaussianNB
from sklearn.naive_bayes import GaussianNB  # import Gaussian Naive Bayes model
NB = GaussianNB()  # create the model
NB.fit(x_train, y_train)  # train the model
predictions = NB.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = NB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(NB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.4322    0.6234    0.5105      1349
           1     0.3416    0.1940    0.2474      1356
           2     0.3272    0.1850    0.2363      1357
           3     0.4457    0.6441    0.5269      1301

    accuracy                         0.4089      5363
   macro avg     0.3867    0.4116    0.3803      5363
weighted avg     0.3860    0.4089    0.3786      5363

[[841 225 123 160]
 [564 263 223 306]
 [350 180 251 576]
 [191 102 170 838]]
Cross Validation Scores are [0.41722595 0.42561521 0.42170022 0.42337808 0.41666667 0.42729306
 0.40962507 0.42137661 0.39955232 0.42809177 0.44127517 0.42449664
 0.4295302  0.39932886 0.41946309 0.39597315 0.42809177 0.40067152
 0.41634024 0.43424734 0.42673378 0.43232662 0.41722595 0.39932886
 0.41498881 0.40771812 0.40962507 0.41242306 0.43200895 0.42025741]
Average Cross Validation score :0.4184193197452644


In [29]:
# support vector machine
from sklearn.svm import LinearSVC  # import linear SVM model
classifier = LinearSVC()  # create the model
classifier.fit(x_train, y_train)  # train the model
y_predict = classifier.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, y_predict, digits=4))  # show test metrics
print(confusion_matrix(y_test, y_predict))  # show test confusion matrix
y_score = classifier.fit(x_train, y_train).decision_function(x_test)  # get decision scores

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, y_predict)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(classifier, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.4308    0.7020    0.5340      1349
           1     0.3320    0.1224    0.1789      1356
           2     0.3220    0.1326    0.1879      1357
           3     0.4079    0.6603    0.5043      1301

    accuracy                         0.4013      5363
   macro avg     0.3732    0.4043    0.3512      5363
weighted avg     0.3727    0.4013    0.3494      5363

[[947 129  65 208]
 [640 166 162 388]
 [392 134 180 651]
 [219  71 152 859]]
Cross Validation Scores are [0.41275168 0.41051454 0.40659955 0.40771812 0.40268456 0.42337808
 0.41130386 0.40962507 0.39115837 0.42977057 0.43344519 0.41219239
 0.41387025 0.39541387 0.42170022 0.40548098 0.4045887  0.38780078
 0.41242306 0.41410185 0.42058166 0.42505593 0.40883669 0.40995526
 0.40995526 0.38758389 0.41522104 0.40514829 0.41354225 0.41410185]
Average Cross Validation score :0.4105501265039954


In [30]:
# AdaBoostClassifier
from sklearn.ensemble import AdaBoostClassifier  # import AdaBoost model
abc = AdaBoostClassifier(n_estimators=50, learning_rate=1)  # create the model
model = abc.fit(x_train, y_train)  # train the model
y_pred = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, y_pred, digits=4))  # show test metrics
print(confusion_matrix(y_test, y_pred))  # show test confusion matrix
y_score = abc.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(abc, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.5268    0.5893    0.5563      1349
           1     0.5019    0.3850    0.4357      1356
           2     0.3243    0.3272    0.3258      1357
           3     0.4810    0.5342    0.5062      1301

    accuracy                         0.4580      5363
   macro avg     0.4585    0.4589    0.4560      5363
weighted avg     0.4582    0.4580    0.4553      5363

[[795 208 201 145]
 [301 522 347 186]
 [239 255 444 419]
 [174  55 377 695]]
Cross Validation Scores are [0.46588367 0.44742729 0.47147651 0.45973154 0.44742729 0.46085011
 0.46054841 0.45831002 0.45775042 0.46838276 0.46644295 0.46252796
 0.45357942 0.43847875 0.45469799 0.45246085 0.45775042 0.45495243
 0.46278679 0.4622272  0.46644295 0.46308725 0.4435123  0.450783
 0.45469799 0.45357942 0.46726357 0.45998881 0.47341914 0.45439284]
Average Cross Validation score :0.458362001730119


In [31]:
# GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier  # import gradient boosting model
GB = GradientBoostingClassifier()  # create the model
model = GB.fit(x_train, y_train)  # train the model
predictions = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = GB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(GB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.6874    0.7042    0.6957      1349
           1     0.5604    0.7353    0.6360      1356
           2     0.4655    0.2837    0.3526      1357
           3     0.6356    0.6718    0.6532      1301

    accuracy                         0.5978      5363
   macro avg     0.5873    0.5987    0.5844      5363
weighted avg     0.5866    0.5978    0.5835      5363

[[950 194  61 144]
 [ 70 997 243  46]
 [185 476 385 311]
 [177 112 138 874]]
Cross Validation Scores are [0.62583893 0.60961969 0.61744966 0.61856823 0.61073826 0.61073826
 0.60716284 0.5954113  0.58981533 0.6144376  0.62751678 0.62472036
 0.59395973 0.61073826 0.61073826 0.6090604  0.60100727 0.58813654
 0.60828204 0.62115277 0.60514541 0.62472036 0.59899329 0.61353468
 0.6163311  0.60514541 0.60324566 0.61219922 0.61891438 0.60716284]
Average Cross Validation score :0.6100161619651749


In [32]:
# XGBoost
from numpy import loadtxt  # import loadtxt from NumPy
from xgboost import XGBClassifier  # import XGBoost model
XGB = XGBClassifier()  # create the model
model = XGB.fit(x_train, y_train)  # train the model
predictions = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = XGB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(XGB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.8099    0.7865    0.7980      1349
           1     0.5612    0.7235    0.6321      1356
           2     0.5813    0.4318    0.4956      1357
           3     0.8034    0.8009    0.8022      1301

    accuracy                         0.6843      5363
   macro avg     0.6890    0.6857    0.6820      5363
weighted avg     0.6876    0.6843    0.6805      5363

[[1061  181   53   54]
 [  69  981  274   32]
 [ 130  472  586  169]
 [  50  114   95 1042]]
Cross Validation Scores are [0.70302013 0.72706935 0.70246085 0.69798658 0.71085011 0.71308725
 0.68830442 0.69054281 0.69837717 0.69949636 0.69183445 0.71029083
 0.69127517 0.700783   0.71196868 0.68847875 0.69669838 0.70509233
 0.71684387 0.70397314 0.69798658 0.69463087 0.70246085 0.70749441
 0.69910515 0.70190157 0.68774482 0.71012871 0.71572468 0.70565193]
Average Cross Validation score :0.7023754395716516


In [33]:
# Bagging
from sklearn.ensemble import BaggingClassifier
BC=BaggingClassifier()
model=BC.fit(x_train, y_train)
predictions = model.predict(x_test)
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,predictions,digits=4))
print(confusion_matrix(y_test,predictions))
y_score = BC.fit(x_train, y_train).predict_proba(x_test)

from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(y_test,predictions)

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import  cross_val_score,KFold
kf = RepeatedStratifiedKFold(n_splits = 10, n_repeats=3, random_state=1)
scores = cross_val_score(BC, x, y, cv = kf)
print("Cross Validation Scores are {}".format(scores))
print("Average Cross Validation score :{}".format(scores.mean()))

              precision    recall  f1-score   support

           0     0.7519    0.7976    0.7741      1349
           1     0.5479    0.6416    0.5910      1356
           2     0.5430    0.4377    0.4847      1357
           3     0.7992    0.7679    0.7832      1301

    accuracy                         0.6599      5363
   macro avg     0.6605    0.6612    0.6583      5363
weighted avg     0.6589    0.6599    0.6568      5363

[[1076  131   83   59]
 [ 132  870  311   43]
 [ 151  463  594  149]
 [  72  124  106  999]]
Cross Validation Scores are [0.69183445 0.68120805 0.67337808 0.68456376 0.68176734 0.68288591
 0.67879127 0.67823167 0.67039731 0.68662563 0.68903803 0.68176734
 0.6689038  0.68176734 0.68847875 0.68512304 0.68606603 0.68774482
 0.68606603 0.66927812 0.66387025 0.68120805 0.70134228 0.69854586
 0.6761745  0.67225951 0.67823167 0.68047006 0.69949636 0.68326805]
Average Cross Validation score :0.6822927790275863
